# 03 · Sri Lankan PII dataset — Colab runner

Runs the pipeline from the `Data` branch. **Everything is saved to Google Drive** in `MyDrive/PII_Project/`
(sources, generated records, checks, review sheets, splits, cost log). Models run on **Vertex AI** and are billed
to your Google Cloud credits, so a **CPU runtime is enough**.

**Before the first run**
1. *Colab Secrets* (key icon on the left): add `GOOGLE_CLOUD_PROJECT` = `numeric-advice-375404` and `HF_TOKEN`;
   turn on *Notebook access* for both.
2. *Once only*: upload `pii_project_upload.zip` (made on your PC with `python -m src.common.pack`) into
   `MyDrive/PII_Project/` — it carries stage 1, your review verdicts, the API cache and the cost log.
   Cell 3 unpacks it and renames it to `.done`.

Re-running is safe: finished records are skipped and cached model answers are never paid twice.

## 1 · Connect Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = "https://github.com/sakuna47/real-time-multilingual-pii-detection.git"
BRANCH = "Data"
REPO = "/content/repo"
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} fetch -q && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!pip install -q -r requirements.txt

## 2 · Session setup (Drive paths, secrets, Vertex login, cache)

In [ ]:
import os, sys
ROOT = "/content/drive/MyDrive/PII_Project"     # all outputs live here
os.makedirs(ROOT, exist_ok=True)
os.environ["PII_DATA_ROOT"] = ROOT
sys.path.insert(0, REPO)

from src.common import colab
colab.setup_env()                                # API cache + Hugging Face downloads on fast local disk
print("secrets:", colab.load_secrets())
colab.vertex_login()                             # sign-in pop-up: use the account that owns the credits
print("unpacked upload:", colab.unpack_upload())  # first run only; None afterwards
print("cache restored:", colab.restore_cache())
sync = colab.start_cache_sync(every_min=5)       # cache is copied to Drive every 5 minutes

In [ ]:
# Check Vertex works and see the remaining budget (estimate from the call log at list price)
!python -m src.llm.smoke --model vertex:gemini-3.5-flash
!python -m src.llm.cost

## 3 · Import sources (once)
Nemotron (~300 MB) and the first WildChat shards are downloaded to Colab's disk; only the filtered records are
saved to Drive. Aya, multiNER and the code-mixing pools came in the upload zip; they are imported only if missing.

In [ ]:
import os
src = f"{ROOT}/sources"
if not os.path.exists(f"{src}/nemotron_pii/manifest.json") or os.path.getsize(f"{src}/nemotron_pii/train.jsonl") < 50_000_000:
    !python -m src.sources.run nemotron
if not os.path.exists(f"{src}/wildchat/manifest.json") or sum(1 for _ in open(f"{src}/wildchat/prompts.jsonl")) < 3000:
    !python -m src.sources.run wildchat --shards 3          # ~4,500 candidate prompts; raise for more
for name, marker in [("aya", "aya/manifest.json"), ("multiner", "multiner/manifest.json"), ("codemix", "codemix/manifest.json")]:
    if not os.path.exists(f"{src}/{marker}"):
        !python -m src.sources.run {name}
!ls {src}

## 4 · Stage 2 generation
`--new N` takes the next N source records that are not done and were not rejected before, so stage 1 is never
redone. Estimated cost with Flex pricing: Nemotron ≈ $2.1 per 1,000 records, WildChat ≈ $3.3 per 1,000 written
records (about two-thirds of WildChat prompts are rejected as unsuitable, and those calls are paid too).
The pipeline refuses to start when the estimated remaining credit is below $1.

In [ ]:
NEMOTRON_NEW = 2500
!python -m src.translate.pipeline --source nemotron --new {NEMOTRON_NEW} --batch-size 6 --workers 8
!python -m src.llm.cost

In [ ]:
WILDCHAT_NEW = 3000     # prompts to try; ~30% become records
AYA_NEW = 200
!python -m src.translate.pipeline --source wildchat --new {WILDCHAT_NEW} --batch-size 6 --workers 8
!python -m src.translate.pipeline --source aya --new {AYA_NEW} --batch-size 6 --workers 8
!python -m src.llm.cost

## 5 · Checks, review sheet, splits

In [ ]:
# Deterministic checks (free). Review verdicts already on records are kept.
!python -m src.validate.checks {ROOT}/generated/sri_lankan

In [ ]:
# Review sheet for new records (already-reviewed ones are left out). Download the HTML from
# MyDrive/PII_Project/review/, open it in your browser, export the CSV, put the CSV back in that folder.
!python -m src.validate.review_export {ROOT}/validated/sri_lankan --per-group 5 --negatives --out {ROOT}/review/stage2_review.html

In [ ]:
# After reviewing: import the CSV(s) from Drive
import glob
csvs = " ".join(f'"{p}"' for p in sorted(glob.glob(f"{ROOT}/review/review_*.csv")))
!python -m src.validate.review_import {ROOT}/validated/sri_lankan {csvs}

In [ ]:
# Train/dev/test by group (variants of one source text never cross splits); multiNER kept as an external test set
!python -m src.pii_data.split {ROOT}/validated/sri_lankan --out {ROOT}/splits/sri_lankan
!python -m src.pii_data.split {ROOT}/sources/multiner --out {ROOT}/splits/multiner --all-test --keep human_verified

## 6 · Finish: cost report and save the cache to Drive

In [ ]:
!python -m src.llm.cost
print("cache saved:", colab.save_cache())
sync.set()